# Exploración y Transformaciones del Dataset de Pokémon

Este notebook muestra ejemplos de **carga, limpieza y transformación** del archivo **`all_pokemon.json`** (incluido en el ZIP) empleando distintas librerías de la ciencia de datos en Python: `pandas`, `numpy`, `geopandas`, y `matplotlib`.

> Ajusta las rutas de los archivos (`zip_path`, etc.) según la ubicación que uses en tu entorno.

In [ ]:
import zipfile, json, pandas as pd, numpy as np

# === 1) Cargar datos desde el ZIP ===
zip_path = '../data/all_pokemon.zip'  # ← Ajusta esta ruta si es necesario
with zipfile.ZipFile(zip_path) as z:
    with z.open('all_pokemon.json') as f:
        raw = json.load(f)

# Convertir dict→DataFrame (cada Pokémon es un dict)
df = pd.DataFrame(raw.values())
print(df.shape)
df.head()

In [ ]:
# === 2) Aplanar el campo 'stats' ===
stats_df = df['stats'].apply(pd.Series)
stats_cols = stats_df[['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']]
stats_cols = stats_cols.rename(columns={
    'special-attack': 'sp_attack',
    'special-defense': 'sp_defense'
})
df = pd.concat([df.drop(columns=['stats']), stats_cols], axis=1)
df.head()

In [ ]:
# === 3) Transformaciones NumPy (BMI e índice de poder) ===
df['altura_m'] = df['height'] / 10  # decímetros → metros
df['peso_kg'] = df['weight'] / 10   # hectogramos → kg
df['indice_corporeo'] = df['peso_kg'] / df['altura_m']**2

# Matriz de stats para operaciones vectorizadas
stats_matrix = df[['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']].to_numpy()
pesos = np.array([1.2, 1.1, 1.1, 1.0, 1.0, 1.3])  # pesos arbitrarios
df['power_index'] = stats_matrix.dot(pesos)
df[['name', 'indice_corporeo', 'power_index']].head()

In [ ]:
# === 4) One‑hot de tipo principal ===
df['primary_type'] = df['types'].apply(lambda lst: lst[0]['type']['name'])
type_ohe = pd.get_dummies(df['primary_type'], prefix='type')
df = pd.concat([df, type_ohe], axis=1)
df.filter(regex='^type_').head()

In [ ]:
import matplotlib.pyplot as plt

# === 5) Altura vs Peso ===
plt.figure(figsize=(6, 4))
plt.scatter(df['altura_m'], df['peso_kg'])
plt.xlabel('Altura (m)')
plt.ylabel('Peso (kg)')
plt.title('Altura vs Peso de Pokémon')
plt.show()

In [ ]:
# === 6) Ejemplo con GeoPandas ===
import geopandas as gpd

print('Placeholder: Carga aquí tu CSV con coordenadas de regiones Pokémon')
# Supongamos que existe "regiones.csv" con columnas: region, lat, lon
# regiones = pd.read_csv('regiones.csv')
# gdf = gpd.GeoDataFrame(regiones, geometry=gpd.points_from_xy(regiones.lon, regiones.lat), crs='EPSG:4326')
# gdf.plot()

In [ ]:
# === 7) Guardar el DataFrame final ===
df.to_parquet('pokemon_limpio.parquet', index=False)
print('✔️  Archivo pokemon_limpio.parquet guardado.')